# 02. Làm sạch dữ liệu VietJobs

**Đầu vào:** `data/raw/VietJobs.csv` (48.092 × 18). **Đầu ra:** `data/processed/vietjobs_clean.parquet`.

Toàn bộ logic nằm trong `src/cleaning.py` (có test trong `tests/test_cleaning.py`). Notebook này chạy pipeline,
trình bày các quyết định làm sạch và kiểm tra kết quả trước/sau.

| Vấn đề | Quyết định |
|---|---|
| Lương "Thoả thuận" lưu là 0 | `salary_negotiable=True`, `salary_avg_num=NaN` |
| `salary_avg` gốc làm tròn xuống (8,5–9,0 → "8 triệu") | `salary_avg_num` = trung điểm thật tính từ chuỗi `salary` |
| Lương < 1 triệu hoặc > 200 triệu/tháng | `salary_outlier=True` (lương giờ/ngày, lỗi nhập) — loại khỏi phân tích lương |
| Kinh nghiệm dạng chữ | `experience_months` (số) + `experience_level` (nhóm có thứ tự) |
| Địa điểm không chuẩn, nhiều tỉnh/tin | `location_primary` (tỉnh đầu, "hà đông"→"hà nội"), `multi_location`, `region` |
| Tin ngoài Việt Nam | loại |
| Tin trùng (cùng tên việc, địa điểm, mô tả) | giữ bản đầu tiên |
| List lưu dạng chuỗi | parse thành list, đếm `n_benefits`, `n_technical_skills` |
| Loại hợp đồng hiếm (< 100 tin) | gộp vào "Khác" |

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.cleaning import RAW_PATH, CLEAN_PATH, clean, parse_salary_avg

raw = pd.read_csv(RAW_PATH)
print(f"Raw: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
(raw.isna().mean() * 100).round(1).sort_values(ascending=False).to_frame("missing_%").head(8)

Raw: 48,092 rows x 18 columns


,missing_%
languages_required,74.4
technical_skills,13.4
working_hours,10.4
soft_skills,5.5
qualifications,0.8
benefits,0.1
location,0.0
job_title,0.0


## Số dòng bị loại theo từng bước

In [2]:
vn = raw[raw["country"] == "Việt Nam"]
steps = pd.DataFrame({
    "bước": ["Dữ liệu gốc", "Chỉ giữ Việt Nam", "Bỏ trùng (job_title, location, description)"],
    "số dòng": [len(raw), len(vn), len(vn.drop_duplicates(["job_title", "location", "description"]))],
})
steps["bị loại"] = steps["số dòng"].shift(1) - steps["số dòng"]
steps

,bước,số dòng,bị loại
0,Dữ liệu gốc,48092,NaN
1,Chỉ giữ Việt Nam,47957,135.0
2,"Bỏ trùng (job_title, location, description)",36457,11500.0


**Nhận xét về trùng lặp:** ~11.500 tin có cùng tên việc, địa điểm và mô tả — chủ yếu là cùng một tin được đăng lại
(phúc lợi chỉ khác cách diễn đạt). Giữ lại sẽ làm các nhà tuyển dụng đăng nhiều lần bị đếm quá mức.

In [3]:
df = clean(raw)
print(f"Clean: {df.shape[0]:,} rows x {df.shape[1]} columns")
summary = pd.Series({
    "% thoả thuận": df["salary_negotiable"].mean() * 100,
    "số tin ngoại lai lương": df["salary_outlier"].sum(),
    "số tin có lương hợp lệ": (~df["salary_negotiable"] & ~df["salary_outlier"]).sum(),
    "trung vị lương (triệu)": df.loc[~df["salary_outlier"], "salary_avg_num"].median(),
}).round(2)
summary

Clean: 36,457 rows x 33 columns


% thoả thuận                 29.18
số tin ngoại lai lương       24.00
số tin có lương hợp lệ    25796.00
trung vị lương (triệu)       13.50
dtype: float64

In [4]:
for col in ["experience_level", "region", "contract_group"]:
    display(df[col].value_counts().to_frame())

,count
experience_level,
1-2 năm,19208
Không yêu cầu,8994
3-5 năm,6290
Dưới 1 năm,1777
Trên 5 năm,188
Không rõ,0


,count
region,
Hà Nội,16149
TP.HCM,13209
Khác,7099


,count
contract_group,
Toàn thời gian,34891
Bán thời gian,657
Thực tập,650
Khác,259


In [5]:
df[["salary_avg_num", "experience_months", "n_benefits", "n_technical_skills"]].describe().round(2)

,salary_avg_num,experience_months,n_benefits,n_technical_skills
count,25820.00,36457.00,36457.00,36457.00
mean,15.53,16.96,6.33,2.66
std,11.77,16.14,2.66,2.23
min,0.10,0.00,0.00,0.00
25%,10.50,6.00,5.00,1.00
50%,13.50,12.00,6.00,2.00
75%,17.50,24.00,8.00,4.00
max,500.00,180.00,35.00,39.00


## Lưu dữ liệu sạch

In [6]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CLEAN_PATH, index=False)
print("Saved:", CLEAN_PATH, f"({CLEAN_PATH.stat().st_size/1e6:.1f} MB)")

Saved: D:\DataScience\CSC17104-final-project\data\processed\vietjobs_clean.parquet (35.9 MB)
